## B_03 · O4: clasificación del circuito (issue #12)
Parte 1: descargar la clasificación individual publicada en la web. Nunca se muestran nombres.

In [1]:
import sys
from pathlib import Path

# Los notebooks están en notebooks/: añadimos la raíz del repo para poder importar src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import pandas as pd
from bs4 import BeautifulSoup
from src.config import DIR_RAW, DIR_INTERIM
from src.obtencion import descargar
from src.carreras import url_pagina, parsear_hay_siguiente

### Cómo es la tabla
Se descarga la página 1 y se miran los nombres de las columnas (solo las clases, sin datos).

In [ ]:
URL_CLASIFICACION = "https://corunacorre.com/es/web/resultado/edicion-63/clasificacion/157"

# Se descarga con la función común (User-Agent, pausa y sin repetir) y se guarda en datos/raw/
html = descargar(url_pagina(URL_CLASIFICACION, 1),
                 destino=DIR_RAW / "clasificacion" / "pagina_1.html")

# Ver como se llaman las columnas de la tabla (solo los nombres de las clases, sin datos)
fila = BeautifulSoup(html, "lxml").find("table").find("tbody").find("tr")
print([td.get("class") for td in fila.find_all("td")])
print("¿Hay página siguiente?:", parsear_hay_siguiente(html))

[['puesto'], ['nombre'], ['apellidos'], ['puntos'], ['desempate1'], ['puesto_sexo'], ['puesto_categoria']]
¿Hay página siguiente?: True


### Descargar todas las páginas
Con la función común `descargar()` (User-Agent, pausa y sin repetir). Se guarda en `datos/interim/clasificacion_publicada.csv` (no se sube: lleva nombres).

In [ ]:
from src.proteccion import quitar_identificadores

# Nombre de la columna en nuestra tabla  -  clase de la celda en la web
COLUMNAS = {
    "posto": "puesto",
    "nome": "nombre",
    "apelidos": "apellidos",
    "puntos": "puntos",
    "desempate1": "desempate1",
    "posto_sexo": "puesto_sexo",
    "posto_categoria": "puesto_categoria",
}


def filas_de(html):
    """Saca las filas de la tabla de una página, como texto."""
    filas = []
    for tr in BeautifulSoup(html, "lxml").find("table").find("tbody").find_all("tr"):
        celdas = {td["class"][0]: td.get_text(" ", strip=True) for td in tr.find_all("td") if td.get("class")}
        if celdas:
            filas.append({col: celdas.get(clase, "") for col, clase in COLUMNAS.items()})
    return filas


# Recorrer las páginas sin saber cuántas hay: se para cuando no hay "Siguiente"
filas, n = [], 1
while True:
    html = descargar(url_pagina(URL_CLASIFICACION, n),
                     destino=DIR_RAW / "clasificacion" / f"pagina_{n}.html")
    filas += filas_de(html)
    if not parsear_hay_siguiente(html):
        break
    n += 1

publicada = pd.DataFrame(filas)
publicada.to_csv(DIR_INTERIM / "clasificacion_publicada.csv", index=False)
print("Páginas:", n, "· filas:", len(publicada))

Páginas: 21 · filas: 515


In [4]:
quitar_identificadores(publicada).head()

,posto,puntos,desempate1,posto_sexo,posto_categoria
0,1,"5,00","6,00",M-1,SUB14M-1
1,2,"5,00","5,00",M-2,SUB16M-1
2,3,"7,00","6,00",M-3,SUB18M-1
3,4,"10,00","5,00",M-4,SUB12M-1
4,5,"10,00","5,00",M-5,SUB16M-2


### Resultado
- 21 páginas y 515 filas (20 × 25 + 15): coherente.
- La clasificación mezcla adultos y menores (SUB10…SUB18). Para comparar con nuestro cálculo habrá que quedarse con las categorías adultas.